# Notebook 06 — Baseline Inference Smoke Test (v1.0.0)

Purpose:
- Load the active baseline model from `models/model_registry.json`
- Build the exact inference feature schema using lookup enrichment
- Run single-request + batch sanity predictions

This notebook is intentionally **not** wiring the FastAPI endpoint yet.

In [7]:
from __future__ import annotations

import json
import datetime as dt
from pathlib import Path

import numpy as np
import pandas as pd
import joblib

pd.set_option('display.max_columns', 200)

# Needed for unpickling: the saved pipeline references this symbol from __main__
def sparse_to_dense(X):
    return X.toarray() if hasattr(X, 'toarray') else X

def find_ml_root(start: Path) -> Path:
    current = start.resolve()
    for _ in range(6):
        if (current / 'models' / 'model_registry.json').exists():
            return current
        current = current.parent
    raise FileNotFoundError('Could not find ml-service root containing models/model_registry.json')

ML_ROOT = find_ml_root(Path.cwd())
REGISTRY_PATH = ML_ROOT / 'models' / 'model_registry.json'
PROCESSED_PATH = ML_ROOT / 'data' / 'processed' / 'processed_data.csv'
LOOKUP_PATH = ML_ROOT / 'data' / 'lookups' / 'car_specs_lookup_full_cleaned.csv'

print('ML_ROOT:', ML_ROOT)
print('REGISTRY_PATH:', REGISTRY_PATH)
print('PROCESSED_PATH:', PROCESSED_PATH)
print('LOOKUP_PATH:', LOOKUP_PATH)

ML_ROOT: /home/mo-seif/Documents/GP/ml-service
REGISTRY_PATH: /home/mo-seif/Documents/GP/ml-service/models/model_registry.json
PROCESSED_PATH: /home/mo-seif/Documents/GP/ml-service/data/processed/processed_data.csv
LOOKUP_PATH: /home/mo-seif/Documents/GP/ml-service/data/lookups/car_specs_lookup_full_cleaned.csv


In [19]:
with open(REGISTRY_PATH, 'r') as f:
    registry = json.load(f)

print('Registry keys:', sorted(registry.keys()))

active_version = registry.get('active_version') or 'v1.0.0'
versions = registry.get('versions') or {}

active_model = registry.get('active_model')
if active_model is None:
    v = versions.get(active_version)
    if not v:
        raise KeyError("Registry missing 'active_model' and versions[active_version] is empty")
    models = v.get('models') or []
    if not models:
        raise KeyError("Registry missing 'active_model' and versions[active_version].models is empty")
    active_model = sorted(models, key=lambda m: m.get('rank', 999))[0]
    print('active_model missing; using versions entry:', active_model.get('model_type'), 'rank', active_model.get('rank'))

active_model_path = ML_ROOT / active_model['pkl_path']

active_preprocessor_meta_path = None
try:
    active_preprocessor_meta_path = ML_ROOT / versions[active_version]['preprocessor']['meta_path']
except Exception:
    candidate = ML_ROOT / 'models' / 'metadata' / 'preprocessor__v1.0.0.json'
    if candidate.exists():
        active_preprocessor_meta_path = candidate
    else:
        raise

print('Active version:', active_version)
print('Active model:', active_model.get('model_type'), 'rank', active_model.get('rank'))
print('Model artifact:', active_model_path)
print('Preprocessor meta:', active_preprocessor_meta_path)

Registry keys: ['active_model', 'active_version', 'versions']
Active version: v1.0.0
Active model: HuberRegressor(tuned) rank 1
Model artifact: /home/mo-seif/Documents/GP/ml-service/models/pickles/Baseline1__huberregressortuned.pkl
Preprocessor meta: /home/mo-seif/Documents/GP/ml-service/models/metadata/preprocessor__v1.0.0.json


In [10]:
model = joblib.load(active_model_path)
print(model)

Pipeline(steps=[('preprocess',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['year', 'car_age',
                                                   'mileage_km',
                                                   'mileage_per_year',
                                                   'engine_cc', 'horsepower',
                                                   'seating_capacity']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleIm

In [11]:
with open(active_preprocessor_meta_path, 'r') as f:
    pre_meta = json.load(f)

numeric_features = list(pre_meta['numeric_features'])
categorical_features = list(pre_meta['categorical_features'])
required_features = numeric_features + categorical_features

print('Numeric features:', numeric_features)
print('Categorical features:', categorical_features)
print('Total required:', len(required_features))

Numeric features: ['year', 'car_age', 'mileage_km', 'mileage_per_year', 'engine_cc', 'horsepower', 'seating_capacity']
Categorical features: ['make', 'model', 'transmission', 'fuel', 'location', 'body_type', 'drivetrain', 'brand_origin', 'car_segment']
Total required: 16


In [12]:
processed = pd.read_csv(PROCESSED_PATH)
lookup = pd.read_csv(LOOKUP_PATH)

print('processed shape:', processed.shape)
print('lookup shape:', lookup.shape)

missing_required = [c for c in required_features if c not in processed.columns]
if missing_required:
    raise ValueError(f'processed_data.csv is missing required features: {missing_required}')

reference_year = int(pd.to_numeric(processed['year'], errors='coerce').dropna().max())
print('reference_year (from processed data):', reference_year)

processed shape: (20461, 18)
lookup shape: (4184, 11)
reference_year (from processed data): 2026


In [13]:
def _mode_or_unknown(series: pd.Series) -> str:
    series = series.dropna().astype(str)
    if series.empty:
        return 'Unknown'
    return series.mode().iloc[0]

def build_features_from_request(
    request: dict,
    lookup_df: pd.DataFrame,
    processed_df: pd.DataFrame,
    *,
    numeric_cols: list[str],
    categorical_cols: list[str],
    reference_year: int,
) -> pd.DataFrame:
    # Map API fields → training feature names
    make = str(request['brand']).strip()
    model_name = str(request['model']).strip()
    year = int(request['year'])
    mileage_km = float(request['mileage_km'])

    base = pd.DataFrame([{
        'make': make,
        'model': model_name,
        'year': year,
        'mileage_km': mileage_km,
        'transmission': str(request['transmission']).strip(),
        'fuel': str(request['fuel']).strip(),
        'location': str(request['location']).strip(),
    }])

    # Enrich from lookup on (make, model, year)
    lk = lookup_df.copy()
    lk['make'] = lk['make'].astype(str).str.strip()
    lk['model'] = lk['model'].astype(str).str.strip()
    lk['year'] = pd.to_numeric(lk['year'], errors='coerce').astype('Int64')

    base['year'] = pd.to_numeric(base['year'], errors='coerce').astype('Int64')
    enriched = base.merge(lk, on=['make', 'model', 'year'], how='left', suffixes=('', '_lk'))

    # Engineer
    year_int = int(enriched.loc[0, 'year'])
    car_age = max(0, reference_year - year_int)
    enriched['car_age'] = float(car_age)
    enriched['mileage_per_year'] = float(mileage_km / max(car_age, 1))

    # Ensure required columns exist
    for col in numeric_cols + categorical_cols:
        if col not in enriched.columns:
            enriched[col] = np.nan

    # Fill missing values using processed-data priors
    priors_num = processed_df[numeric_cols].apply(pd.to_numeric, errors='coerce').median(numeric_only=True)
    for col in numeric_cols:
        enriched[col] = pd.to_numeric(enriched[col], errors='coerce')
        if pd.isna(enriched.loc[0, col]):
            enriched.loc[0, col] = float(priors_num[col]) if col in priors_num.index else 0.0

    for col in categorical_cols:
        if pd.isna(enriched.loc[0, col]) or str(enriched.loc[0, col]).strip() == '':
            enriched.loc[0, col] = _mode_or_unknown(processed_df[col]) if col in processed_df.columns else 'Unknown'
        enriched[col] = enriched[col].astype(str)

    # Return in the exact required feature order
    return enriched[numeric_cols + categorical_cols].copy()

In [14]:
example_request = {
    'brand': 'Toyota',
    'model': 'Corolla',
    'year': 2018,
    'mileage_km': 85000,
    'transmission': 'Automatic',
    'fuel': 'petrol',
    'location': 'Cairo',
}

X_new = build_features_from_request(
    example_request,
    lookup,
    processed,
    numeric_cols=numeric_features,
    categorical_cols=categorical_features,
    reference_year=reference_year,
)

X_new

,year,car_age,mileage_km,mileage_per_year,engine_cc,horsepower,seating_capacity,make,model,transmission,fuel,location,body_type,drivetrain,brand_origin,car_segment
0,2018,8.0,85000.0,10625.0,1600,122,5,Toyota,Corolla,Automatic,petrol,Cairo,Sedan,FWD,japanese,family


In [18]:
pred_raw = float(model.predict(X_new)[0])

# Heuristic: if prediction looks like log-price, convert to EGP
pred_egp = float(np.exp(pred_raw)) if pred_raw < 50 else pred_raw

neg_low = pred_egp * 0.85
neg_high = pred_egp * 1.15

predicted_at = dt.datetime.now(dt.UTC).isoformat().replace('+00:00', 'Z')

print('pred_raw:', pred_raw)
print('fair_price_egp:', round(pred_egp, 2))
print('negotiation_range:', round(neg_low, 2), '-', round(neg_high, 2))
print('predicted_at:', predicted_at)
print('model_version:', active_version)

pred_raw: 13.715241975753731
fair_price_egp: 904594.43
negotiation_range: 768905.26 - 1040283.59
predicted_at: 2026-05-07T00:35:40.328812Z
model_version: v1.0.0


In [ ]:
# Sanity check: compare prediction to similar cars in processed data
make = X_new.loc[0, 'make']
model_name = X_new.loc[0, 'model']
subset = processed[(processed['make'] == make) & (processed['model'] == model_name)]

if subset.empty:
    print('No exact make/model matches found in processed_data.csv for distribution check.')
else:
    prices = pd.to_numeric(subset['price_egp'], errors='coerce').dropna()
    q05, q10, q30, q50, q75, q90, q95 = prices.quantile([0.05, 0.10, 0.30, 0.50, 0.75, 0.90, 0.95]).tolist()
    print(f'n_similar={len(prices)} | p05={q05:,.0f} | p10={q10:,.0f} \
            | p30={q30:,.0f} | median={q50:,.0f} | p75={q75:,.0f} | p90={q90:,.0f} | p95={q95:,.0f}')
    print('prediction:', f'{pred_egp:,.0f}')
    if pred_egp < q05 or pred_egp > q95:
        print('⚠️  Prediction is outside [p05, p95] for same make/model (may still be OK if trim differs).')

n_similar=344 | p05=205,750 | p10=260,000 | p30=569,000 | median=797,500 | p75=1,000,000 | p90=1,300,000 | p95=1,450,000
prediction: 904,594
